## Section D - email.csv

In [1]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import TimestampType
from pyspark.sql.window import Window
from pyspark.sql.functions import col, lower, trim, when, regexp_extract

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("CERT Analysis") \
    .master("local[*]") \
    .config("spark.driver.host", "127.0.0.1") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .config("spark.executorEnv.SPARK_LOCAL_IP", "127.0.0.1") \
    .getOrCreate()


In [3]:
# utils.py contains all helpers
from helpers import code_quality, parse_date, letters_cleaning

In [4]:
email_raw = spark.read.csv('../data/email.csv', header=True, inferSchema=True)

In [5]:
email_raw.show(3)

+--------------------+-------------------+-------+-------+--------------------+--------------------+--------------------+--------------------+-----+-----------+--------------------+
|                  id|               date|   user|     pc|                  to|                  cc|                 bcc|                from| size|attachments|             content|
+--------------------+-------------------+-------+-------+--------------------+--------------------+--------------------+--------------------+-----+-----------+--------------------+
|{C0K9-N2TG12WK-37...|01/02/2010 06:50:45|LRG0155|PC-0450|Rafael.H.Mccall@n...|                NULL|Lysandra_Guerrero...|Lysandra_Guerrero...|29918|          0|apple lot mundane...|
|{F5Y9-A8ON18GE-55...|01/02/2010 07:05:31|KAB0942|PC-8686|Ali.Cole.Mclaughl...|Karly.Amity.Bentl...|                NULL|Karly.Amity.Bentl...|22235|          0|again 15 bought a...|
|{I8R9-T8NW12WZ-66...|01/02/2010 07:06:46|KAB0942|PC-8686|Brent.Colorado.Sa...|           

In [6]:
email_raw.count()

2733360

File `email.csv` Contains: email sending/receiving events. <br>
Columns: id, date, user, pc, to, cc, bcc, from, size, attachments, content <br>
Meaning for GNN: user -> recipient edge (especially external),
large attachments or email to multiple recipients = signal

In [18]:
# cleaning
email = email_raw \
    .dropDuplicates() \
    .filter(F.col("user").isNotNull()) \
    .filter(F.col("from").isNotNull())

In [19]:
# date-parsing
email = parse_date(email)

In [ ]:
# In CERT internal domains are of format: dtaa.com.
# Let's mark users that sent emails outside of the organization.
email = email.withColumn(
    "external_recipient",
    ~F.col("to").contains("dtaa.com")  
)

In [17]:
email.show()

+--------------------+-------------------+-------+-------+--------------------+--------------------+--------------------+--------------------+-----+-----------+--------------------+-------------------+----+-----+---+----+---------------+-----------------+------------------+
|                  id|               date|   user|     pc|                  to|                  cc|                 bcc|                from| size|attachments|             content|          timestamp|year|month|day|hour|day_of_the_week|not_working_hours|external_recipient|
+--------------------+-------------------+-------+-------+--------------------+--------------------+--------------------+--------------------+-----+-----------+--------------------+-------------------+----+-----+---+----+---------------+-----------------+------------------+
|{A1W1-S8VO35UB-98...|01/02/2010 10:14:56|DMR0116|PC-2025|tjn1239@optonline...|Xanthus_David@msn...|Celeste_C_Spears@...|deborah_rocha@gma...|52070|          0|similar distinc